In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Dataframe5').getOrCreate()

C:\Users\LENOVO\AppData\Roaming\Python\Python314\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
spark

In [3]:
training = spark.read.csv('test1.csv', header=True, inferSchema=True)

In [4]:
training.show()

+---------+---+----------+------+
|     Name|Age|Experience|Salary|
+---------+---+----------+------+
|    Krish| 31|        10| 30000|
|Sudhanshu| 30|         8| 25000|
|    Sunny| 29|         4| 20000|
|     Paul| 24|         3| 20000|
|   Harsha| 21|         1| 15000|
|  Shubham| 23|         2| 18000|
+---------+---+----------+------+



In [5]:
training.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [6]:
training.columns

['Name', 'Age', 'Experience', 'Salary']

In [7]:
from pyspark.ml.feature import VectorAssembler

In [9]:
featureassembler = VectorAssembler(inputCols=['Age','Experience'],outputCol='independent feature')

In [10]:
output = featureassembler.transform(training)

In [11]:
output.show()

+---------+---+----------+------+-------------------+
|     Name|Age|Experience|Salary|independent feature|
+---------+---+----------+------+-------------------+
|    Krish| 31|        10| 30000|        [31.0,10.0]|
|Sudhanshu| 30|         8| 25000|         [30.0,8.0]|
|    Sunny| 29|         4| 20000|         [29.0,4.0]|
|     Paul| 24|         3| 20000|         [24.0,3.0]|
|   Harsha| 21|         1| 15000|         [21.0,1.0]|
|  Shubham| 23|         2| 18000|         [23.0,2.0]|
+---------+---+----------+------+-------------------+



In [12]:
finalized_data = output.select(['independent feature','Salary'])

In [13]:
finalized_data.show()

+-------------------+------+
|independent feature|Salary|
+-------------------+------+
|        [31.0,10.0]| 30000|
|         [30.0,8.0]| 25000|
|         [29.0,4.0]| 20000|
|         [24.0,3.0]| 20000|
|         [21.0,1.0]| 15000|
|         [23.0,2.0]| 18000|
+-------------------+------+



In [16]:
from pyspark.ml.regression import LinearRegression
# train test split
train_data,test_data = finalized_data.randomSplit([0.75,0.25])
regressor = LinearRegression(featuresCol = 'independent feature', labelCol = 'Salary')
regressor = regressor.fit(train_data)

In [22]:
regressor.coefficients

DenseVector([-90.5483, 1608.7819])

In [23]:
regressor.intercept

16079.136690647425

In [24]:
pred_results = regressor.evaluate(test_data)

In [25]:
pred_results.predictions.show()

+-------------------+------+-----------------+
|independent feature|Salary|       prediction|
+-------------------+------+-----------------+
|         [23.0,2.0]| 18000|17214.09079632846|
+-------------------+------+-----------------+



In [26]:
pred_results.meanAbsoluteError,pred_results.meanSquaredError

(785.909203671541, 617653.2764156357)